In [1]:
import numpy as np 
import torch
import pandas as pd
import h5py 
from importlib import reload
import pickle
from IPython.display import display, Audio
from lightning_scripts import jsinV3DataLoader_precombined_batched as jsinv3

In [2]:
reload(jsinv3)
MatchedAudiosetBatched = jsinv3.MatchedAudiosetBatched
# h5_path = '/mnt/home/jfeather/ceph/data/training_datasets_audio/audioset_dataframes/sr20000/sr20000_unbalanced_train_segments_raw_exclude_speech_and_only_music.pdh5' 
h5_path = "/mnt/home/igriffith/ceph/datasets/audioset/unbalanced_train_segments.hdf5"
dataset = MatchedAudiosetBatched(h5_path, target_keys=['noise/labels'], batch_size=4)


In [3]:
h5 = h5py.File(h5_path)
print(h5.keys())
# print(h5['labels'][0])
h5.close()

<KeysViewHDF5 ['labels', 'wav']>


In [4]:
batch_ix = 200
SR = 20_000
[output_11, output_12, output_21, output_22], [target_11, target_12, target_21 , target_22] = dataset[batch_ix]

display(Audio(output_11[0], rate=SR))
display(Audio(output_12[0], rate=SR))
display(Audio(output_21[0], rate=SR))
display(Audio(output_22[0], rate=SR))

In [5]:
target_11['noise/labels'].shape

torch.Size([4, 527])

### Test lightning module  

In [ ]:
import yaml
import lightning as L
import lightning_scripts.lightning_ssl_matched_audioset_only as lightning
reload(jsinv3)

reload(lightning)

LitAudioSSL = lightning.LitAudioSSL
## init config. Will be yaml eventually, but start as dict 

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01_audioset_only.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 0
config['hparas']['batch_size'] = 4
config['hparas']['global_batch_size'] = 4
config['num_gpus'] = 0
model = LitAudioSSL(config)
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1
                    )
trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name            | Type                       | Params | Mode 
----------------------------------------------------------------------

Rank 0 N training batches 251124


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Rank 0 N validation batches 5081


/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Rank 0 N training batches 251124


/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Training: |          | 0/? [00:00<?, ?it/s]